# Saved-cells deep dive

Per-cell analyses on `test_predictions.csv` (see
`src/python/classifier_training/README.md`'s "Saving per-cell test predictions"),
joined against spatial coordinates (`patch_coordinates.h5`), H&E imagery, and RNA
expression (`adata.h5ad`) -- all under `XENIUM_PROCESSED_OUTPUT_ROOT` /
`WSI_RAW_ROOT` / `WSI_CONVERTED_ROOT`. See `classification_results_deep_dive.ipynb`
for the metrics-only comparisons that don't need any of this.

**This needs the cluster mount.** `~/Documents/mount_leomed` was empty/unmounted
when this notebook was written, and none of the four `test_predictions.csv` pairs
below existed locally yet either. Every cell here was written against the
documented data layout and reviewed carefully, but **none of it has been executed
against real data** -- mount the drive, sync/generate the `test_predictions.csv`
files, then run top to bottom and expect to fix the odd path/column-name mismatch.
`classification_results_deep_dive.ipynb`, by contrast, *was* executed end-to-end
against real local data.

## What's covered

1. Neighbor-cell-type baseline (112px / 224px radius) vs. the model's own accuracy.
2. Top-k accuracy.
3. H&E crop gallery, one grid per (true, predicted) confusion pair.
4. Spatially-misclassified regions, incl. cells wrong regardless of which
   condition (CLS/central, masked/unmasked) is used.
5. **Disagreement toolkit** -- generic: given any two aligned prediction
   conditions, split into the 4 correct/wrong quadrants, show class composition,
   plot example cells, and check whether the swapped cells are predictable from
   spatial context alone. Applied to CLS-vs-central (`UNI2_448_resized`) and
   masked-vs-unmasked (`PhikonV2_448`).
6. RNA-expression-vs-context divergence for the masking disagreement set.
7. Does that divergence predict *where* masking helps or hurts.

## Reusing this notebook

`SAVED_PREDICTIONS` (§1) is the single place new `(model, embeddings)` pairs get
added -- everything downstream re-runs unchanged. The disagreement toolkit in §5
(`align_conditions` / `disagreement_summary` / `context_predictability`) takes any
two already-loaded, coordinate-attached prediction dataframes, so a third pairing
(e.g. a future context-size comparison) is one function call, not new code.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu
from sklearn.metrics import f1_score, balanced_accuracy_score
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def find_repo_root(start: Path) -> Path:
    # Jupyter's cwd is wherever the server was launched from (usually notebooks/),
    # not necessarily the repo root -- same pattern as
    # Bulk_Deconvolution_HNE_Prior/notebooks/diagnose_patch_coordinates.ipynb.
    for d in [start, *start.parents]:
        if (d / "src" / "python" / "code_configs").is_dir():
            return d
    raise RuntimeError(f"could not locate repo root above {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT = {REPO_ROOT}")


def load_dotenv(path: Path) -> None:
    """Minimal `.env` loader (equivalent to `set -a; source .env; set +a`) so the
    env-var-backed paths in src.python.code_configs.paths resolve inside a plain
    Jupyter kernel, which doesn't inherit a shell-sourced .env automatically.
    """
    if not path.exists():
        print(f"[no .env at {path} -- env-var-backed paths below will raise until you create one]")
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())


load_dotenv(REPO_ROOT / ".env")

from src.python.code_configs import paths as P  # noqa: E402

OKABE_ITO = ["#E69F00", "#56B4E9", "#009E73", "#F0E442",
             "#0072B2", "#D55E00", "#CC79A7", "#000000"]
mpl.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 12,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG_DIR = Path("figures_saved_cells")
FIG_DIR.mkdir(exist_ok=True)


def savefig(fig, name):
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    print(f"saved -> {path}")

## Config

In [ ]:
TEST_PREDICTIONS_ROOT = Path("../outputs_classifier_colon")  # same tree as classification_results_deep_dive.ipynb
CORRECTION_NAME = "raw"
MAPPING = "simplified_broad"
MATCHING = "all_cells"
CLASS_ORDER = ["Epithelial", "Lymphoid", "Malignant", "Myeloid", "Stromal"]

# (model_name, embeddings) pairs with test_predictions.csv saved -- add rows here as
# more get saved; every section below discovers/skips per-pair automatically.
SAVED_PREDICTIONS = [
    {"model_name": "PhikonV2_448_masked", "embeddings": "nucleus"},
    {"model_name": "PhikonV2_448",        "embeddings": "nucleus"},
    {"model_name": "UNI2_448_resized",    "embeddings": "cls"},
    {"model_name": "UNI2_448_resized",    "embeddings": "nucleus"},
]

# Quick reachability check for the two external data roots this notebook needs, so a
# missing mount fails here with a clear message instead of a confusing error four
# cells down.
for name, root in [("XENIUM_PROCESSED_OUTPUT_ROOT", P.XENIUM_PROCESSED_OUTPUT_ROOT),
                    ("WSI_RAW_ROOT", P.WSI_RAW_ROOT)]:
    ok = Path(root).is_dir() and any(Path(root).iterdir())
    print(f"{'OK ' if ok else 'MISSING '} {name} -> {root}")
    if not ok:
        print("    (mount ~/Documents/mount_leomed and/or check .env if this looks wrong)")

## 1. Loading `test_predictions.csv`

Schema (see `classifier_training/README.md`): `cell_id, wsi, true_label,
predicted_label, prob_<class>..., correct` -- one row per test cell, from the
best (val-selected) trial of that split.

In [ ]:
def load_test_predictions(root: Path, model_name: str, embeddings: str,
                           correction_name: str = CORRECTION_NAME, mapping: str = MAPPING,
                           matching: str = MATCHING, include_same_wsi: bool = False) -> pd.DataFrame:
    """Load and concatenate every LOSO split's test_predictions.csv for one
    (model_name, embeddings) condition, tagged with split_idx / is_same_wsi. Missing
    splits/files are skipped with a printed warning rather than raising -- check
    `.empty` on the result before using it.
    """
    base = root / model_name / correction_name / mapping / matching / embeddings
    if not base.exists():
        print(f"[missing] {base}")
        return pd.DataFrame()

    split_dirs = sorted(base.glob("split_*"))
    if include_same_wsi and (base / "same_wsi_split").exists():
        split_dirs.append(base / "same_wsi_split")

    frames = []
    for split_dir in split_dirs:
        csv_path = split_dir / "test_predictions.csv"
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        df["correct"] = df["correct"].astype(bool)
        is_same_wsi = split_dir.name == "same_wsi_split"
        df["split_label"] = split_dir.name
        df["split_idx"] = None if is_same_wsi else int(split_dir.name.removeprefix("split_"))
        df["is_same_wsi"] = is_same_wsi
        df["model_name"] = model_name
        df["embeddings"] = embeddings
        frames.append(df)

    if not frames:
        print(f"[no test_predictions.csv found under] {base}")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


predictions: dict[tuple[str, str], pd.DataFrame] = {}
for spec in SAVED_PREDICTIONS:
    key = (spec["model_name"], spec["embeddings"])
    predictions[key] = load_test_predictions(TEST_PREDICTIONS_ROOT, **spec)
    n = len(predictions[key])
    print(f"{key}: {n} test-cell rows" + ("" if n else " -- nothing loaded (see [missing]/[no ...] above)"))

## 2. Spatial coordinates

From `patch_coordinates.h5` (H&E **pixel** space -- `x_start`/`y_start` are the
top-left corner of each cell's 224x224 patch, so cell center =
`x_start+112, y_start+112`; see the top-level README's "Input:
`patch_coordinates.h5`"). Deliberately *not* `adata.h5ad`'s
`x_centroid`/`y_centroid`, which are in Xenium's own micron coordinate frame --
using pixel space here means the 112px/224px radii requested below are literal
pixel radii, matching the patch-size convention used everywhere else in this
codebase, with no unit conversion to get wrong.

In [ ]:
_COORDS_CACHE: dict[str, pd.DataFrame] = {}


def load_patch_coordinates(wsi: str) -> pd.DataFrame:
    path = P.XENIUM_PROCESSED_OUTPUT_ROOT / wsi / "patch_coordinates.h5"
    with h5py.File(path, "r") as f:
        raw_ids = f["cell_id"][:]
        cell_id = np.array([c.decode() if isinstance(c, bytes) else c for c in raw_ids])
        x_start = f["x_start"][:]
        y_start = f["y_start"][:]
    out = pd.DataFrame({"cell_id": cell_id, "x_start": x_start, "y_start": y_start})
    out["x_centroid_px"] = out["x_start"] + 112
    out["y_centroid_px"] = out["y_start"] + 112
    return out


def get_patch_coordinates(wsi: str) -> pd.DataFrame:
    if wsi not in _COORDS_CACHE:
        _COORDS_CACHE[wsi] = load_patch_coordinates(wsi)
    return _COORDS_CACHE[wsi]


def attach_pixel_coords(df: pd.DataFrame) -> pd.DataFrame:
    """Join x_centroid_px/y_centroid_px onto a test_predictions-shaped dataframe
    (needs cell_id + wsi), from patch_coordinates.h5, per WSI (so cell_id only needs
    to be unique *within* a WSI, not globally).
    """
    if df.empty:
        return df
    out = []
    for wsi, group in df.groupby("wsi"):
        coords = get_patch_coordinates(wsi).set_index("cell_id")[["x_centroid_px", "y_centroid_px"]]
        out.append(group.join(coords, on="cell_id"))
    return pd.concat(out, ignore_index=True)

## 3. Neighbor-cell-type baseline (112px / 224px radius)

A purely spatial baseline: for each test cell, take the majority **ground-truth**
label among same-WSI cells within `radius_px` (self excluded), and score that
against `true_label` the same way the model itself is scored. This is a
diagnostic upper bound, not a real classifier -- ground-truth neighbor labels
aren't available at inference time -- but it answers "how much of the model's
accuracy could spatial homogeneity alone already explain".

In [ ]:
def neighbor_majority_predictions(df_with_coords: pd.DataFrame, radius_px: float,
                                   label_col: str = "true_label") -> pd.Series:
    """Majority `label_col` among same-WSI neighbors within radius_px (self
    excluded), per cell. NaN where a cell has no neighbor in radius. Returned Series
    shares df_with_coords's index.
    """
    preds = pd.Series(index=df_with_coords.index, dtype=object)
    for wsi, group in df_with_coords.groupby("wsi"):
        xy = group[["x_centroid_px", "y_centroid_px"]].values
        tree = cKDTree(xy)
        neighbor_lists = tree.query_ball_point(xy, r=radius_px)
        labels = group[label_col].values
        for local_i, (row_idx, neighbors) in enumerate(zip(group.index, neighbor_lists)):
            others = [n for n in neighbors if n != local_i]
            if not others:
                continue
            vals, counts = np.unique(labels[others], return_counts=True)
            preds.loc[row_idx] = vals[np.argmax(counts)]
    return preds


def evaluate_neighbor_baseline(df_with_coords: pd.DataFrame, radii_px=(112, 224)) -> pd.DataFrame:
    rows = []
    for r in radii_px:
        pred = neighbor_majority_predictions(df_with_coords, r)
        mask = pred.notna()
        rows.append({
            "radius_px": r,
            "coverage": mask.mean(),
            "macro_f1": f1_score(df_with_coords.loc[mask, "true_label"], pred[mask], average="macro", zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(df_with_coords.loc[mask, "true_label"], pred[mask]),
            "accuracy": (df_with_coords.loc[mask, "true_label"].values == pred[mask].values).mean(),
        })
    rows.append({
        "radius_px": "model (own prediction)",
        "coverage": 1.0,
        "macro_f1": f1_score(df_with_coords["true_label"], df_with_coords["predicted_label"], average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(df_with_coords["true_label"], df_with_coords["predicted_label"]),
        "accuracy": df_with_coords["correct"].mean(),
    })
    return pd.DataFrame(rows)


for key in [("UNI2_448_resized", "cls"), ("UNI2_448_resized", "nucleus")]:
    df = predictions.get(key, pd.DataFrame())
    if df.empty:
        continue
    df_coords = attach_pixel_coords(df)
    print(f"\n=== {key}: neighbor-majority baseline vs. model ===")
    display(evaluate_neighbor_baseline(df_coords).round(4))

## 4. Top-k accuracy

Is the true label among the `k` highest-probability classes, using the
`prob_<class>` columns.

In [ ]:
def topk_accuracy(df: pd.DataFrame, k: int, class_order=CLASS_ORDER) -> float:
    # Assumes true_label is always one of class_order -- true for simplified_broad
    # (Unknown cells are dropped before training/eval, see the top-level README's
    # "Cell-type mappings"). A row outside class_order silently scores as a miss.
    prob_cols = [f"prob_{c}" for c in class_order]
    probs = df[prob_cols].values
    topk_idx = np.argsort(-probs, axis=1)[:, :k]
    true_idx = df["true_label"].map({c: i for i, c in enumerate(class_order)}).values
    return float(np.mean([true_idx[i] in topk_idx[i] for i in range(len(df))]))


def topk_recall_per_class(df: pd.DataFrame, k: int, class_order=CLASS_ORDER) -> pd.Series:
    """Per true-class top-k accuracy -- which classes still get missed even at k>1."""
    prob_cols = [f"prob_{c}" for c in class_order]
    probs = df[prob_cols].values
    topk_idx = np.argsort(-probs, axis=1)[:, :k]
    true_idx = df["true_label"].map({c: i for i, c in enumerate(class_order)}).values
    hit = np.array([true_idx[i] in topk_idx[i] for i in range(len(df))])
    return pd.Series(hit, index=df.index).groupby(df["true_label"]).mean().reindex(class_order)


def topk_table(predictions: dict, ks=(1, 2, 3), class_order=CLASS_ORDER) -> pd.DataFrame:
    rows = []
    for (model, emb), df in predictions.items():
        if df.empty:
            continue
        rows.append({"model_name": model, "embeddings": emb,
                      **{f"top{k}_accuracy": topk_accuracy(df, k, class_order) for k in ks}})
    return pd.DataFrame(rows)


display(topk_table(predictions))

for key, df in predictions.items():
    if df.empty:
        continue
    print(f"\n=== {key}: top-2 recall per true class ===")
    display(topk_recall_per_class(df, k=2).round(4))

## 5. Confusion-pair cell gallery

H&E crops for a sample of cells from each `(true, predicted)` misclassification
pair that actually occurs, so you can eyeball e.g. "Malignant predicted as
Epithelial" cells directly.

In [ ]:
import openslide as Openslide  # noqa: E402

_WSI_CACHE: dict[str, "Openslide.OpenSlide"] = {}


def wsi_path(wsi: str) -> Path:
    # Tries the converted/registered naming first, then raw -- see the top-level
    # README's "Input: patch_coordinates.h5" / "converted: True" note. Adjust if your
    # WSIs use a different naming convention than these two.
    candidates = [
        P.WSI_CONVERTED_ROOT / f"{wsi}_registered_HE.ome.tif",
        P.WSI_RAW_ROOT / f"{wsi}_he_image.ome.tif",
    ]
    for c in candidates:
        if Path(c).exists():
            return Path(c)
    raise FileNotFoundError(f"no WSI file found for {wsi!r} among {candidates}")


def get_wsi(wsi: str) -> "Openslide.OpenSlide":
    if wsi not in _WSI_CACHE:
        _WSI_CACHE[wsi] = Openslide.OpenSlide(str(wsi_path(wsi)))
    return _WSI_CACHE[wsi]


def read_cell_crop(wsi: str, x_centroid_px: float, y_centroid_px: float, size: int = 224):
    slide = get_wsi(wsi)
    x0, y0 = int(x_centroid_px - size / 2), int(y_centroid_px - size / 2)
    return slide.read_region((x0, y0), 0, (size, size)).convert("RGB")


def plot_cell_grid(df_with_coords: pd.DataFrame, n: int = 6, size: int = 224, seed: int = 0, title: str = ""):
    """Grid of H&E crops for a sample of df_with_coords (needs wsi, x_centroid_px,
    y_centroid_px, true_label, predicted_label).
    """
    if df_with_coords.empty:
        print(f"[no cells to plot] {title}")
        return
    sample = df_with_coords.sample(min(n, len(df_with_coords)), random_state=seed)
    ncols = min(len(sample), 6)
    nrows = int(np.ceil(len(sample) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.9 * nrows))
    axes = np.atleast_1d(axes).flatten()
    for ax in axes[len(sample):]:
        ax.axis("off")
    for ax, (_, row) in zip(axes, sample.iterrows()):
        try:
            crop = read_cell_crop(row["wsi"], row["x_centroid_px"], row["y_centroid_px"], size)
            ax.imshow(crop)
        except Exception as e:
            ax.text(0.5, 0.5, f"load error:\n{e}", ha="center", va="center", fontsize=7, wrap=True)
        ax.set_title(f"{row['true_label']} -> {row['predicted_label']}", fontsize=9)
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def confusion_pair_gallery(df_with_coords: pd.DataFrame, n_per_pair: int = 4, class_order=CLASS_ORDER, seed: int = 0):
    """One small grid per (true, predicted) pair that actually occurs among
    df_with_coords's misclassified cells, in class_order x class_order.
    """
    wrong = df_with_coords[~df_with_coords["correct"]]
    for true_c in class_order:
        for pred_c in class_order:
            if true_c == pred_c:
                continue
            pair = wrong[(wrong["true_label"] == true_c) & (wrong["predicted_label"] == pred_c)]
            if pair.empty:
                continue
            plot_cell_grid(pair, n=n_per_pair, seed=seed,
                            title=f"true={true_c}, predicted={pred_c}  (n={len(pair)})")


# Pick whichever condition you want to inspect -- defaults to the nucleus read-out.
key = ("UNI2_448_resized", "nucleus")
df = predictions.get(key, pd.DataFrame())
if not df.empty:
    confusion_pair_gallery(attach_pixel_coords(df), n_per_pair=4)
else:
    print(f"[skip] {key} not loaded")

## 6. Spatially misclassified regions

Where errors cluster within a WSI, plus (since strict LOSO tests each WSI
exactly once, so "wrong in every split" isn't meaningful the way it would be
under k-fold) the cross-**condition** generalization of the same question:
which cells are wrong regardless of which read-out (CLS/central,
masked/unmasked, ...) is used.

In [ ]:
def plot_misclassification_map(df_coords: pd.DataFrame, wsi: str, bin_px: int = 500, min_cells_per_bin: int = 5):
    sub = df_coords[df_coords["wsi"] == wsi]
    if sub.empty:
        print(f"[no cells for {wsi}]")
        return None
    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

    axes[0].scatter(sub.loc[sub["correct"], "x_centroid_px"], sub.loc[sub["correct"], "y_centroid_px"],
                     s=2, color="#56B4E9", alpha=0.4, label="correct")
    axes[0].scatter(sub.loc[~sub["correct"], "x_centroid_px"], sub.loc[~sub["correct"], "y_centroid_px"],
                     s=4, color="#D55E00", alpha=0.7, label="misclassified")
    axes[0].invert_yaxis(); axes[0].set_aspect("equal"); axes[0].legend(markerscale=4)
    axes[0].set_title(f"{wsi}: correct vs. misclassified")

    x_bins = np.arange(sub["x_centroid_px"].min(), sub["x_centroid_px"].max() + bin_px, bin_px)
    y_bins = np.arange(sub["y_centroid_px"].min(), sub["y_centroid_px"].max() + bin_px, bin_px)
    total, _, _ = np.histogram2d(sub["x_centroid_px"], sub["y_centroid_px"], bins=[x_bins, y_bins])
    wrong, _, _ = np.histogram2d(sub.loc[~sub["correct"], "x_centroid_px"], sub.loc[~sub["correct"], "y_centroid_px"],
                                  bins=[x_bins, y_bins])
    rate = np.divide(wrong, total, out=np.full_like(wrong, np.nan), where=total >= min_cells_per_bin)
    im = axes[1].imshow(rate.T, origin="lower", cmap="Reds", vmin=0, vmax=1,
                         extent=[x_bins[0], x_bins[-1], y_bins[0], y_bins[-1]])
    axes[1].set_aspect("equal")
    fig.colorbar(im, ax=axes[1], label=f"misclassification rate per {bin_px}px bin (>={min_cells_per_bin} cells)")
    axes[1].set_title(f"{wsi}: binned error rate")
    fig.tight_layout()
    plt.show()
    return rate


def misclassification_maps_all_wsis(df_coords: pd.DataFrame, bin_px: int = 500):
    for wsi in sorted(df_coords["wsi"].unique()):
        plot_misclassification_map(df_coords, wsi, bin_px)


def cells_wrong_in_all_conditions(dfs_coords: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """dfs_coords: {condition_label: df_with_coords}, each with cell_id, wsi,
    x_centroid_px, y_centroid_px, true_label, correct. Returns the union of cells
    (by cell_id+wsi) with one correct__<label> bool column per condition plus
    wrong_in_all (true only for cells present and wrong in *every* condition).
    """
    merged = None
    for label, df in dfs_coords.items():
        d = df[["cell_id", "wsi", "x_centroid_px", "y_centroid_px", "true_label", "correct"]].copy()
        d = d.rename(columns={"correct": f"correct__{label}"})
        merged = d if merged is None else merged.merge(
            d.drop(columns=["x_centroid_px", "y_centroid_px", "true_label"]), on=["cell_id", "wsi"], how="outer"
        )
    correct_cols = [c for c in merged.columns if c.startswith("correct__")]
    merged["wrong_in_all"] = merged[correct_cols].notna().all(axis=1) & (~merged[correct_cols].fillna(True)).all(axis=1)
    return merged


key = ("UNI2_448_resized", "nucleus")
df = predictions.get(key, pd.DataFrame())
if not df.empty:
    misclassification_maps_all_wsis(attach_pixel_coords(df))
else:
    print(f"[skip] {key} not loaded")

cls_df = attach_pixel_coords(predictions.get(("UNI2_448_resized", "cls"), pd.DataFrame()))
nucleus_df = attach_pixel_coords(predictions.get(("UNI2_448_resized", "nucleus"), pd.DataFrame()))
if not cls_df.empty and not nucleus_df.empty:
    wrong_everywhere = cells_wrong_in_all_conditions({"cls": cls_df, "nucleus": nucleus_df})
    print(f"{wrong_everywhere['wrong_in_all'].sum()} / {len(wrong_everywhere)} cells wrong in both cls and nucleus")
    plot_misclassification_map(wrong_everywhere.assign(correct=~wrong_everywhere["wrong_in_all"]),
                                wsi=wrong_everywhere["wsi"].mode().iat[0])

## 7. Disagreement toolkit

Generic: given two aligned prediction conditions (same test cells, two
different read-outs), split into the four correct/wrong quadrants, show what
cell types make up each quadrant, plot examples, and check whether the
swapped-outcome cells are predictable from spatial context alone (reusing the
§3 neighbor-majority baseline). Applied below to CLS-vs-central
(`UNI2_448_resized`) and masked-vs-unmasked (`PhikonV2_448`).

In [ ]:
def align_conditions(df_a: pd.DataFrame, df_b: pd.DataFrame, label_a: str, label_b: str) -> pd.DataFrame:
    """Inner-join two coordinate-attached test_predictions dataframes on
    (cell_id, wsi) -- only cells evaluated as test cells under *both* conditions are
    kept. Adds a `quadrant` column: both_correct / both_wrong / <label_a>_only_correct
    / <label_b>_only_correct.
    """
    keep = ["cell_id", "wsi", "true_label", "predicted_label", "correct"]
    a = df_a[keep].rename(columns={c: f"{c}__{label_a}" for c in keep if c not in ("cell_id", "wsi")})
    b = df_b[keep].rename(columns={c: f"{c}__{label_b}" for c in keep if c not in ("cell_id", "wsi")})
    merged = a.merge(b, on=["cell_id", "wsi"], how="inner")
    assert (merged[f"true_label__{label_a}"] == merged[f"true_label__{label_b}"]).all(), \
        "true_label disagrees between conditions for a shared cell_id -- check both used the same mapping"
    merged["true_label"] = merged[f"true_label__{label_a}"]

    correct_a, correct_b = merged[f"correct__{label_a}"], merged[f"correct__{label_b}"]
    merged["quadrant"] = np.select(
        [correct_a & correct_b, (~correct_a) & (~correct_b), correct_a & (~correct_b), (~correct_a) & correct_b],
        ["both_correct", "both_wrong", f"{label_a}_only_correct", f"{label_b}_only_correct"],
    )

    coord_cols = [c for c in ("x_centroid_px", "y_centroid_px") if c in df_a.columns]
    if coord_cols:
        merged = merged.merge(df_a[["cell_id", "wsi"] + coord_cols], on=["cell_id", "wsi"], how="left")
    return merged


def disagreement_summary(aligned: pd.DataFrame, class_order=CLASS_ORDER):
    print(aligned["quadrant"].value_counts())
    fig, ax = plt.subplots(figsize=(6, 4))
    ct = pd.crosstab(aligned["true_label"], aligned["quadrant"], normalize="columns").reindex(class_order)
    ct.plot(kind="bar", ax=ax)
    ax.set_ylabel("proportion of quadrant's cells")
    ax.set_title("True cell-type composition of each disagreement quadrant")
    fig.tight_layout()
    plt.show()
    return ct


def context_predictability(quadrant_df: pd.DataFrame, df_coords_full: pd.DataFrame, radius_px: float = 224) -> float:
    """Fraction of quadrant_df's cells whose true_label matches the ground-truth
    neighbor-majority vote (radius_px, from §3) computed against the *full* WSI cell
    population in df_coords_full -- not just the quadrant subset, so neighbor context
    isn't artificially restricted to already-selected cells. Diagnostic only (uses
    ground-truth neighbor labels): answers "could spatial context alone explain this
    disagreement", not "would a real model get it right".
    """
    full = df_coords_full.reset_index(drop=True)
    preds = neighbor_majority_predictions(full, radius_px)
    lookup = full[["cell_id", "wsi"]].assign(_pred=preds.values)
    merged = quadrant_df[["cell_id", "wsi", "true_label"]].merge(lookup, on=["cell_id", "wsi"], how="left")
    return float((merged["_pred"] == merged["true_label"]).mean())


# ---- CLS vs. central (UNI2_448_resized) ----
if not cls_df.empty and not nucleus_df.empty:
    aligned_cls_central = align_conditions(cls_df, nucleus_df, "cls", "nucleus")
    print("=== CLS vs. central (UNI2_448_resized) ===")
    disagreement_summary(aligned_cls_central)

    cls_only = aligned_cls_central[aligned_cls_central["quadrant"] == "cls_only_correct"]
    nucleus_only = aligned_cls_central[aligned_cls_central["quadrant"] == "nucleus_only_correct"]
    plot_cell_grid(nucleus_only, n=6, title="correct w/ central token, wrong w/ CLS")
    plot_cell_grid(cls_only, n=6, title="correct w/ CLS, wrong w/ central token")

    full_pool = pd.concat([cls_df, nucleus_df]).drop_duplicates(["cell_id", "wsi"])
    print(f"context-predictability (radius=224px), cls-only-correct: {context_predictability(cls_only, full_pool):.3f}")
    print(f"context-predictability (radius=224px), nucleus-only-correct: {context_predictability(nucleus_only, full_pool):.3f}")

# ---- Masked vs. unmasked (PhikonV2_448) ----
masked_df = attach_pixel_coords(predictions.get(("PhikonV2_448_masked", "nucleus"), pd.DataFrame()))
unmasked_df = attach_pixel_coords(predictions.get(("PhikonV2_448", "nucleus"), pd.DataFrame()))
if not masked_df.empty and not unmasked_df.empty:
    aligned_mask = align_conditions(unmasked_df, masked_df, "unmasked", "masked")
    print("\n=== Masked vs. unmasked (PhikonV2_448) ===")
    disagreement_summary(aligned_mask)

    broken_by_masking = aligned_mask[aligned_mask["quadrant"] == "unmasked_only_correct"]   # correct unmasked, wrong masked
    fixed_by_masking = aligned_mask[aligned_mask["quadrant"] == "masked_only_correct"]       # correct masked, wrong unmasked
    plot_cell_grid(broken_by_masking, n=6, title="correct unmasked, wrong when masked")
    plot_cell_grid(fixed_by_masking, n=6, title="correct masked, wrong when unmasked")

## 8. RNA expression vs. context, for the masking disagreement set

"Are cells that were correct unmasked but wrong when masked showing different
RNA expression from their spatial context?" -- for each cell, cosine distance
between its log1p-normalized expression and the mean expression of its `k=30`
nearest spatial neighbors (same convention as `deepspot_training`'s
neighbor-aggregation, see that README). Higher = more transcriptionally
different from its immediate neighborhood.

In [ ]:
import anndata as ad  # noqa: E402
import scanpy as sc  # noqa: E402

_ADATA_CACHE: dict[str, "ad.AnnData"] = {}


def load_adata(wsi: str) -> "ad.AnnData":
    """Load (and cache) one WSI's Xenium AnnData, log1p-normalized in place (.X).
    obs['cell_id'] matches test_predictions.csv's cell_id.
    """
    if wsi not in _ADATA_CACHE:
        path = P.XENIUM_PROCESSED_OUTPUT_ROOT / wsi / "adata.h5ad"
        adata = ad.read_h5ad(path)
        adata.X = adata.layers["counts"].copy()
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        _ADATA_CACHE[wsi] = adata
    return _ADATA_CACHE[wsi]


def rna_context_divergence(wsi: str, k_neighbors: int = 30) -> pd.DataFrame:
    """Per-cell RNA-context divergence for every cell in one WSI that has both an
    adata entry and a patch_coordinates entry. Returns a cell_id-indexed DataFrame
    with `divergence` -- join onto any predictions dataframe by cell_id (per-WSI, see
    attach_rna_divergence) afterwards.
    """
    adata = load_adata(wsi)
    coords = get_patch_coordinates(wsi).set_index("cell_id")
    common = coords.index.intersection(adata.obs["cell_id"])
    if len(common) < k_neighbors + 1:
        return pd.DataFrame(columns=["divergence"])

    barcode_to_row = {b: i for i, b in enumerate(adata.obs["cell_id"].values)}
    rows = [barcode_to_row[c] for c in common]
    X = adata.X[rows]
    X = np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)
    xy = coords.loc[common, ["x_centroid_px", "y_centroid_px"]].values

    tree = cKDTree(xy)
    _, idx = tree.query(xy, k=k_neighbors + 1)  # column 0 is the point itself (dist 0)
    neighbor_mean = X[idx[:, 1:]].mean(axis=1)

    own_norm = np.linalg.norm(X, axis=1)
    neigh_norm = np.linalg.norm(neighbor_mean, axis=1)
    denom = own_norm * neigh_norm
    cos_sim = np.divide((X * neighbor_mean).sum(axis=1), denom,
                         out=np.full(len(common), np.nan), where=denom > 0)
    return pd.DataFrame({"divergence": 1 - cos_sim}, index=pd.Index(common, name="cell_id"))


def attach_rna_divergence(df: pd.DataFrame, k_neighbors: int = 30) -> pd.DataFrame:
    if df.empty:
        return df
    out = []
    for wsi, group in df.groupby("wsi"):
        div = rna_context_divergence(wsi, k_neighbors)
        out.append(group.join(div, on="cell_id"))
    return pd.concat(out, ignore_index=True)


if not masked_df.empty and not unmasked_df.empty:
    unmasked_div = attach_rna_divergence(unmasked_df)
    aligned_mask_div = align_conditions(unmasked_div, masked_df, "unmasked", "masked").merge(
        unmasked_div[["cell_id", "wsi", "divergence"]], on=["cell_id", "wsi"], how="left"
    )

    groups_to_compare = {
        "correct in both": aligned_mask_div.loc[aligned_mask_div["quadrant"] == "both_correct", "divergence"].dropna(),
        "correct unmasked, wrong masked": aligned_mask_div.loc[
            aligned_mask_div["quadrant"] == "unmasked_only_correct", "divergence"].dropna(),
    }
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot(groups_to_compare.values(), labels=groups_to_compare.keys())
    ax.set_ylabel("RNA-context divergence (1 - cosine sim to k=30 neighbor mean)")
    ax.set_title("Do cells that break under masking show higher RNA-context divergence?")
    fig.tight_layout()
    savefig(fig, "rna_divergence_broken_by_masking")
    plt.show()

    a, b = groups_to_compare["correct in both"], groups_to_compare["correct unmasked, wrong masked"]
    if len(a) > 5 and len(b) > 5:
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        print(f"Mann-Whitney U: p={p:.4g} (n_both_correct={len(a)}, n_broken_by_masking={len(b)})")
else:
    print("[skip] masked/unmasked PhikonV2 predictions not both loaded")

## 9. Does RNA-context divergence predict where masking helps?

Split cells by RNA-context divergence (default: each WSI's own median, so it's
a relative "more/less context-typical than usual for this tissue" split, not
a fixed global cutoff -- swap `DIVERGENCE_THRESHOLD` for a fixed float to test
a specific value instead) and compare masked-vs-unmasked accuracy within each
bin.

In [ ]:
DIVERGENCE_THRESHOLD = "median"  # or a fixed float, e.g. 0.3


def divergence_bin(df: pd.DataFrame, threshold=DIVERGENCE_THRESHOLD) -> np.ndarray:
    cutoff = df.groupby("wsi")["divergence"].transform("median") if threshold == "median" else threshold
    return np.where(df["divergence"] >= cutoff, "high_divergence", "low_divergence")


if not masked_df.empty and not unmasked_df.empty:
    unmasked_div = attach_rna_divergence(unmasked_df)
    unmasked_div = unmasked_div[unmasked_div["divergence"].notna()].copy()
    unmasked_div["divergence_bin"] = divergence_bin(unmasked_div)

    aligned_bins = align_conditions(unmasked_div, masked_df, "unmasked", "masked").merge(
        unmasked_div[["cell_id", "wsi", "divergence_bin"]], on=["cell_id", "wsi"], how="inner"
    )
    summary = aligned_bins.groupby("divergence_bin").apply(lambda d: pd.Series({
        "n": len(d),
        "unmasked_accuracy": d["correct__unmasked"].mean(),
        "masked_accuracy": d["correct__masked"].mean(),
        "masked_minus_unmasked": d["correct__masked"].mean() - d["correct__unmasked"].mean(),
    }))
    display(summary.round(4))
else:
    print("[skip] masked/unmasked PhikonV2 predictions not both loaded")

## Next steps / how to extend

- Add a new saved-prediction pair: append to `SAVED_PREDICTIONS` (§1), rerun.
- Add a new pairwise comparison (e.g. a future context-size disagreement):
  `align_conditions(df_a, df_b, "a", "b")` + `disagreement_summary` +
  `context_predictability` (§7) work on any two coordinate-attached prediction
  dataframes, not just the two pairs run above.
- `_ADATA_CACHE` / `_WSI_CACHE` / `_COORDS_CACHE` are module-level dicts, so
  reopening a cell later in the session reuses already-loaded WSIs/AnnDatas
  instead of re-reading from the mount.